### Loading 

In [ ]:
from pathlib import Path

import mgnify_methods.paper_modules as pm
from mgnify_methods.taxonomy import (
    pivot_taxonomic_data,
    aggregate_by_taxonomic_level,
)

from mgnify_methods.taxonomy import (
    remove_singletons_per_sample,
    prevalence_cutoff_abund,
)

from mgnify_methods.metacomp.transforms import (
    apply_transform_method,
)

In [2]:
root_dir = Path().resolve().parent
root_dir

WindowsPath('C:/Users/David Palecek/Documents/Python_projects/emobon/emobon-basic-models')

In [3]:
contig_path = root_dir / "configs" / "model_test.json"
CONFIG = pm.config_setup(root_dir, contig_path)

INFO | mgnify_methods.paper_modules | Directory ready: C:\Users\David Palecek\Documents\Python_projects\emobon\emobon-basic-models\outputs\analysis_20260305_1208
INFO | mgnify_methods.paper_modules | Directory ready: C:\Users\David Palecek\Documents\Python_projects\emobon\emobon-basic-models\analysis_cache
INFO | mgnify_methods.paper_modules | Configuration saved to: C:\Users\David Palecek\Documents\Python_projects\emobon\emobon-basic-models\outputs\analysis_20260305_1208\analysis_config.json
INFO | mgnify_methods.paper_modules | Logger configured to write to C:\Users\David Palecek\Documents\Python_projects\emobon\emobon-basic-models\outputs\analysis_20260305_1208\analysis.log


### Load cache if available 

In [4]:
from utils.io import load_preprocessed_cache

In [5]:
try:
    preprocess_tables, emobon_meta = load_preprocessed_cache(root_dir, CONFIG)
except FileNotFoundError:
    print('needs manual processing')

Loaded preprocessing cache from: C:\Users\David Palecek\Documents\Python_projects\emobon\emobon-basic-models\analysis_cache\model_testing


In [4]:
# loads both ssu and emobon metadata, 
abundance_emobon, emobon_meta = pm.load_emobon(root_dir, ret='ssu')
abundance_emobon.shape, emobon_meta.shape

c:\Users\David Palecek\Documents\Python_projects\emobon\emobon-basic-models\.venv\Lib\site-packages\momics\metadata.py:263: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata["replicate_info"] = (


((111320, 9), (181, 100))

In [5]:
abundance_emobon = pm.process_emobon_data(abundance_emobon, CONFIG)
abundance_emobon.shape

(7131, 181)

In [6]:
# turn abundance into taxonomy for cleaning lineaages
taxonomy_table = pm.clean_abundance_table(abundance_emobon, CONFIG)

tax_level = CONFIG['taxonomy']['analysis_level']
abundance_table = aggregate_by_taxonomic_level(
    taxonomy_table.df,
    level=tax_level,
    dropna=CONFIG['preprocessing']['dropna']
)
del taxonomy_table

INFO | mgnify_methods.tables.tables | Converting source: tax_mgnify_raw
WARNING | mgnify_methods.tables.tables | DataFrame is abundance without ncbi_tax_id data.
INFO | mgnify_methods.tables.tables | self.source before conversion to Taxonomy table: tax_mgnify_raw
INFO | mgnify_methods.tables.tables | Converting source: tax_processed
INFO | mgnify_methods.tables.sources.converters | Abundance dtype after conversion: uint16
INFO | mgnify_methods.tables.tables | Validating source: tax_processed
INFO | mgnify_methods.paper_modules | Taxonomy table ready: (111320, 9)


In [7]:
abundance_table = pivot_taxonomic_data(abundance_table)
type(abundance_table)

pandas.core.frame.DataFrame

## Processing of the raw abundance table
- remove singletons
- CLR

In [8]:
abundance_table_beta = remove_singletons_per_sample(abundance_table, skip_columns=0)

abundance_table_beta = prevalence_cutoff_abund(
        abundance_table, 
        percent=CONFIG['preprocessing']['prevalence_cutoff'],
        skip_columns=0
    )

preprocess_tables = {'emobon': abundance_table}

## Data transformation for beta diversity
if CONFIG['transform']['enabled']:
    transformed_tables = {}
    for sample_type, table_or_dict in preprocess_tables.items():
        if isinstance(table_or_dict, dict):
            transformed_tables[sample_type] = {
                tax_level: apply_transform_method(df, CONFIG)
                for tax_level, df in table_or_dict.items()
            }
        else:
            transformed_tables[sample_type] = apply_transform_method(table_or_dict, CONFIG)

    preprocess_tables = transformed_tables

INFO | mgnify_methods.metacomp.transforms | 
=== Applying transform clr ===


Removed 10 singleton features. Shape reduced from (221, 181) → (211, 181)
Prevalence cutoff at 0.5% (max threshold 333.105): reduced from (221, 181) to (66, 181)


## Modelling

In [6]:
from emobon_models.modeling_config import modeling_config_from_analysis
from emobon_models.modeling_runner import run_group_loocv_with_mlflow

In [7]:
# filter metadata manually
emobon_meta_in = emobon_meta[CONFIG['modeling']['metadata_cols']]

In [8]:

modeling_config = modeling_config_from_analysis(CONFIG)
abundance_for_model = preprocess_tables["emobon"]

modeling_results = run_group_loocv_with_mlflow(
    metadata_df=emobon_meta_in,
    abundance_df=abundance_for_model,
    config=modeling_config,
)

modeling_results["summary_metrics"]


observatory ID observatory ID
BPNS        23
NRMCB       22
ROSKOGO     20
RFormosa    18
VB          16
AAOT        15
IUIEilat    12
PiEGetxo    12
EMT21       12
OSD74       11
MBAL4        8
ESC68N       6
OOB          3
HCMR-1       3
Name: count, dtype: int64
month name month name
Dec    54
Aug    49
Oct    46
Jun    18
Nov     8
Jul     6
Name: count, dtype: int64
tidal stage tidal stage
no_tide       62
low_tide      34
ebb_tide      33
flood_tide    27
high_tide     23
NA             2
Name: count, dtype: int64
longitude longitude
 14.250000    22
-7.969250     18
 7.317000     16
 2.808331     16
 12.508333    15
-3.968333     14
-8.798500     12
-3.014639     12
 34.916667    12
-8.666639     11
-4.217000      8
 2.808330      7
-3.866000      6
 17.125619     6
 3.135278      3
 25.278761     3
Name: count, dtype: int64
latitude latitude
51.433331    23
40.800139    22
37.005639    18
43.683000    16
45.314167    15
48.771667    14
42.201944    12
43.338583    12
29.500000 

c:\Users\David Palecek\Documents\Python_projects\emobon\emobon-basic-models\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/03/05 12:08:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/05 12:08:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/05 12:08:35 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


,mae_mean,mae_std,rmse_mean,rmse_std
0,3.598921,0.513574,7.158597,0.790428


In [7]:
emobon_meta.head()

,alkalinity (mEq/l),alkalinity method,ammonium (umol/l),ammonium method,bacterial production (ug/m3/d),bacterial production method,biomass (g/l),biomass method,chemical administration,chlorophyll (ug/l),...,water current (m3/s),water current method,year,month,month name,day,season,replicate info,study_tag,sample_type
source material ID,,,,,,,,,,,,,,,,,,,,,
EMOBON_OOB_So_1,NaN,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,2021,6,Jun,8,Spring,OOB_soft_sediment_2021-06-08_NA,EMO-BON,prok
EMOBON_ESC68N_Wa_1,NaN,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,0.821,...,NaN,NaN,2021,8,Aug,30,Summer,ESC68N_water_column_2021-08-30_3-200,EMO-BON,prok
EMOBON_ESC68N_Wa_2,NaN,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,0.821,...,NaN,NaN,2021,8,Aug,30,Summer,ESC68N_water_column_2021-08-30_3-200,EMO-BON,prok
EMOBON_ESC68N_Wa_3,NaN,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,0.821,...,NaN,NaN,2021,8,Aug,30,Summer,ESC68N_water_column_2021-08-30_0.2-3,EMO-BON,prok
EMOBON_ESC68N_Wa_4,NaN,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,0.821,...,NaN,NaN,2021,8,Aug,30,Summer,ESC68N_water_column_2021-08-30_0.2-3,EMO-BON,prok


In [9]:
abundance_for_model.head()

source material ID,EMOBON_AAOT_Wa_1,EMOBON_AAOT_Wa_2,EMOBON_AAOT_Wa_22,EMOBON_AAOT_Wa_26,EMOBON_AAOT_Wa_27,EMOBON_AAOT_Wa_41,EMOBON_AAOT_Wa_42,EMOBON_AAOT_Wa_46,EMOBON_AAOT_Wa_47,EMOBON_AAOT_Wa_6,...,EMOBON_VB_Wa_4,EMOBON_VB_Wa_41,EMOBON_VB_Wa_42,EMOBON_VB_Wa_43,EMOBON_VB_Wa_44,EMOBON_VB_Wa_5,EMOBON_VB_Wa_93,EMOBON_VB_Wa_94,EMOBON_VB_Wa_96,EMOBON_VB_Wa_97
taxonomic_concat,,,,,,,,,,,,,,,,,,,,,
sk__Archaea;k__;p__;c__;o__;f__;g__;s__,-2.537975,-2.537975,-2.537975,-2.537975,-2.537975,-2.537975,-2.537975,-2.537975,-2.537975,-2.537975,...,-2.537975,-2.537975,-2.537975,-2.537975,-2.537975,-2.537975,-2.537975,-2.537975,-2.537975,-2.537975
sk__Archaea;k__unclassified_Crenarchaeota;p__Crenarchaeota;c__;o__;f__;g__;s__,-0.972570,-0.972570,-0.972570,-0.972570,-0.972570,-0.972570,-0.972570,-0.972570,-0.972570,-0.972570,...,-0.972570,-0.972570,-0.972570,-0.972570,-0.972570,-0.972570,-0.972570,-0.972570,-0.972570,-0.972570
sk__Archaea;k__unclassified_Euryarchaeota;p__Euryarchaeota;c__;o__;f__;g__;s__,-12.369128,15.251843,-12.369128,-12.369128,-12.369128,-12.369128,12.543792,15.375222,14.972877,-12.369128,...,-12.369128,-12.369128,-12.369128,-12.369128,-12.369128,-12.369128,-12.369128,-12.369128,-12.369128,-12.369128
sk__Archaea;k__unclassified_Thaumarchaeota;p__Thaumarchaeota;c__;o__;f__;g__;s__,-7.635912,-7.635912,-7.635912,-7.635912,-7.635912,-7.635912,-7.635912,-7.635912,-7.635912,-7.635912,...,-7.635912,-7.635912,-7.635912,-7.635912,-7.635912,-7.635912,-7.635912,-7.635912,-7.635912,-7.635912
sk__Bacteria;k__;p__;c__;o__;f__;g__;s__,0.760431,1.110797,1.184824,0.421568,-0.091122,0.367625,-0.241858,-0.094860,-0.658228,0.691211,...,0.264620,0.269835,0.633249,0.903373,0.528491,0.510254,-0.252763,-0.061708,1.178555,0.645828


In [11]:
[k for k in abundance_for_model.index if 'p__;' in k]

['sk__Archaea;k__;p__;c__;o__;f__;g__;s__',
 'sk__Bacteria;k__;p__;c__;o__;f__;g__;s__',
 'sk__Eukaryota;k__;p__;c__;o__;f__;g__;s__']

In [12]:
filtered = abundance_for_model[~abundance_for_model.index.str.contains('p__;')]

In [13]:
[k for k in filtered.index if 'p__;' in k]

[]

In [14]:
abundance_for_model.shape, filtered.shape

((66, 181), (63, 181))

In [12]:
emobon_meta.columns

Index(['alkalinity (mEq/l)', 'alkalinity method', 'ammonium (umol/l)',
       'ammonium method', 'bacterial production (ug/m3/d)',
       'bacterial production method', 'biomass (g/l)', 'biomass method',
       'chemical administration', 'chlorophyll (ug/l)', 'chlorophyll method',
       'collection_date', 'conductivity (mS/cm)', 'conductivity method',
       'density (g/m3)', 'density method', 'sampling depth (m)',
       'dissolved carbon dioxide (umol/l)', 'dissolved carbon dioxide method',
       'dissolved inorganic carbon (ug/l)',
       'dissolved inorganic carbon method',
       'dissolved organic carbon (umol/l)', 'dissolved organic carbon method',
       'dissolved organic nitrogen (ug/l)',
       'dissolved organic nitrogen method', 'dissolved oxygen (mmol/kg)',
       'dissolved oxygen method', 'visible waveband radiance (uE/m2/s)',
       'downward PAR method', 'environment (biome)', 'environment (feature)',
       'environment (material)', 'environmental package', 'countr

In [13]:
emobon_meta['observatory ID'].value_counts()

observatory ID
BPNS        23
NRMCB       22
ROSKOGO     20
RFormosa    18
VB          16
AAOT        15
IUIEilat    12
PiEGetxo    12
EMT21       12
OSD74       11
MBAL4        8
ESC68N       6
OOB          3
HCMR-1       3
Name: count, dtype: int64

In [10]:
for column in emobon_meta.columns:
    print(column, emobon_meta[column].value_counts(dropna=False))

alkalinity (mEq/l) alkalinity (mEq/l)
NaN    181
Name: count, dtype: int64
alkalinity method alkalinity method
NaN    181
Name: count, dtype: int64
ammonium (umol/l) ammonium (umol/l)
NaN       131
0.1300      4
0.5300      4
0.4100      4
0.3600      4
0.2400      4
0.0044      4
0.8600      4
0.6600      4
5.8800      4
0.1400      4
2.2500      4
0.5400      3
1.7300      2
4.5700      1
Name: count, dtype: int64
ammonium method ammonium method
NA                                                                                                                                                                                                                         152
segmented continuous-flow autoanalyzer                                                                                                                                                                                      14
Spectrophotometry                                                                                     